In [5]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [6]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to C:\Users\HP
[nltk_data]     POWER\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [8]:
# printing the stopwords in english
print(stopwords.words('english'))

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

# Data pre_processing

In [16]:
# loading  the dataset to a pandas DataFrame
news_dataset = pd.read_csv('Fake.csv')
news_dataset2 = pd.read_csv('True.csv')

In [17]:
news_dataset.shape

(23481, 4)

In [18]:
news_dataset2.shape

(21417, 4)

In [11]:
# print the first 5 rows of the dataframe
news_dataset.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [19]:
news_dataset2.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [12]:
# counting the number of missing value
news_dataset.isnull().sum()

title      0
text       0
subject    0
date       0
dtype: int64

In [20]:
news_dataset2.isnull().sum()

title      0
text       0
subject    0
date       0
dtype: int64

In [13]:
# Check what the 'subject' column contains (it is a topic, not a real/fake label)
news_dataset['subject'].value_counts()

subject
News               9050
politics           6841
left-news          4459
Government News    1570
US_News             783
Middle-east         778
Name: count, dtype: int64

In [21]:
news_dataset2['subject'].value_counts()

subject
politicsNews    11272
worldnews       10145
Name: count, dtype: int64

In [22]:
# Load each file separately
fake_df = pd.read_csv('Fake.csv')
true_df = pd.read_csv('True.csv')

In [24]:
# Add a label column to each file BEFORE merging
# 1 = real news, 0 = fake news
true_df['label'] = 1
fake_df['label'] = 0
news_dataset = pd.concat([true_df, fake_df], ignore_index=True)

In [25]:
X = news_dataset.drop(columns=['label'])
Y = news_dataset['label']

In [26]:
X = news_dataset['text']
Y = news_dataset['label']

In [27]:
news_dataset['label'].value_counts()

label
0    23481
1    21417
Name: count, dtype: int64

In [28]:
# Remove "CITY (Reuters) - " from the start of articles
X = news_dataset['text'].str.replace(r'^.*?\(Reuters\)\s*-?\s*', '', regex=True)
Y = news_dataset['label']

In [30]:
#stemming it the process of reducing a word to its root
port_stem = PorterStemmer()
stop_words = set(stopwords.words('english'))  # built once, fast lookup

def stemming(content):
    # keep only letters
    stemmed_content = re.sub('[^a-zA-Z]', ' ', content)
    # lowercase and split into words
    stemmed_content = stemmed_content.lower().split()
    # stem each word, skipping stopwords
    stemmed_content = [port_stem.stem(w) for w in stemmed_content if w not in stop_words]
    # join back into one string
    return ' '.join(stemmed_content)

In [31]:
# Remove "CITY (Reuters) - " only if it appears within the first 60 characters
news_dataset['text'] = news_dataset['text'].str.replace(
    r'^[^()]{0,60}\(Reuters\)\s*-?\s*', '', regex=True
)

# Verify the leak is gone (both should be near 0)
news_dataset['text'].str.contains('Reuters').groupby(news_dataset['label']).mean()

label
0    0.013160
1    0.234954
Name: text, dtype: float64

In [32]:
%time news_dataset['text'].head(500).apply(stemming)

CPU times: total: 4.28 s
Wall time: 4.45 s


0      head conserv republican faction u congress vot...
1      transgend peopl allow first time enlist u mili...
2      special counsel investig link russia presid tr...
3      trump campaign advis georg papadopoulo told au...
4      presid donald trump call u postal servic frida...
                             ...                        
495    follow statement post verifi twitter account u...
496    leandra english legal battl trump administr ac...
497    u senat elizabeth warren said tuesday doubt le...
498    top two republican u congress said lead democr...
499    u congress careen toward major deadlin republi...
Name: text, Length: 500, dtype: str

In [33]:
news_dataset['content'] = news_dataset['text'].apply(stemming)

X = news_dataset['content'].values
Y = news_dataset['label'].values

In [34]:
# Drop the stemmed form of "Reuters" from the cleaned text
news_dataset['content'] = news_dataset['content'].str.replace(r'\breuter\b', '', regex=True)

# Check: both should be 0.0
news_dataset['content'].str.contains('reuter').groupby(news_dataset['label']).mean()

# Rebuild X and Y from the fixed column
X = news_dataset['content'].values
Y = news_dataset['label'].values

In [35]:
# converting the textual data to numerical data
vectorizer = TfidfVectorizer()
vectorizer.fit(X)

X = vectorizer.transform(X)

In [36]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6756878 stored elements and shape (44898, 89564)>
  Coords	Values
  (0, 619)	0.025394996673518258
  (0, 710)	0.037246167201424436
  (0, 821)	0.044720588672501824
  (0, 1281)	0.02773351156331895
  (0, 1430)	0.11701653544836126
  (0, 2281)	0.02571204252815086
  (0, 2296)	0.0464750115965213
  (0, 2688)	0.05284997999702733
  (0, 3581)	0.03434346409814526
  (0, 3585)	0.030114794657806164
  (0, 4057)	0.033887680724581475
  (0, 4405)	0.06773435449275514
  (0, 5338)	0.01888947127232749
  (0, 5673)	0.05530539840551143
  (0, 6296)	0.034248555469207305
  (0, 6737)	0.030277542945904987
  (0, 7570)	0.02794659673082375
  (0, 7583)	0.0323933163804719
  (0, 7643)	0.0752902617741111
  (0, 7670)	0.05867301604801549
  (0, 8834)	0.037458586161402574
  (0, 8868)	0.0290072858036266
  (0, 8926)	0.04910422762824623
  (0, 9224)	0.05840761997626934
  (0, 9894)	0.03255591830767073
  :	:
  (44897, 81070)	0.03417962356851589
  (44897, 81320)	0.019810459

# Splitting the dataset to traning and test data

In [37]:
# now traning
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size= 0.2, stratify=Y, random_state=2)

# Traning the Model: Logistic Regression

In [38]:
model = LogisticRegression()

In [39]:
model.fit(X_train, Y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

# Now let Evaluate

accuracy score

In [40]:
# accuracy score on the traning data
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)

In [41]:
print('Accuracy score of the tranining data : ', training_data_accuracy)

Accuracy score of the tranining data :  0.9855225792081964


In [42]:
# accuracy score on the test data
X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)

In [43]:
print('Accuracy score of the test data : ', test_data_accuracy)

Accuracy score of the test data :  0.9770601336302895


# Now let make a prediction system

In [57]:
X_new = X_test[0]

prediction = model.predict(X_new)
print(prediction)

if (prediction[0]==0):
    print('The news is Real')
else:
    print('The news is Fake')

[1]
The news is Fake


In [58]:
print(Y_test[0])

1
